In [0]:
# CommerceLake - Bronze Initial Ingestion

CATALOG = "commercelake"
BRONZE_SCHEMA = "bronze"

BASE_PATH = "/Volumes/commercelake/landing/source_data"

BATCH_ID = "batch_001"

SOURCE_CONFIG = {
    "customers": {
        "format": "csv",
        "filename": "customers.csv",
        "target_table": "raw_customers",
        "expected_count": 1000
    },
    "products": {
        "format": "csv",
        "filename": "products.csv",
        "target_table": "raw_products",
        "expected_count": 200
    },
    "orders": {
        "format": "json",
        "filename": "orders.json",
        "target_table": "raw_orders",
        "expected_count": 5000
    },
    "order_items": {
        "format": "parquet",
        "filename": "order_items.parquet",
        "target_table": "raw_order_items",
        "expected_count": 10055
    },
    "payments": {
        "format": "json",
        "filename": "payments.json",
        "target_table": "raw_payments",
        "expected_count": 5000
    }
}

print("=" * 70)
print("COMMERCELAKE - BRONZE INGESTION CONFIGURATION")
print("=" * 70)

print(f"Catalog: {CATALOG}")
print(f"Target schema: {BRONZE_SCHEMA}")
print(f"Batch ID: {BATCH_ID}")

for entity, config in SOURCE_CONFIG.items():

    config["source_path"] = (
        f"{BASE_PATH}/{entity}/{BATCH_ID}/{config['filename']}"
    )

    config["target_path"] = (
        f"{CATALOG}.{BRONZE_SCHEMA}.{config['target_table']}"
    )

    print(f"\nEntity: {entity}")
    print(f"Source format: {config['format']}")
    print(f"Source path: {config['source_path']}")
    print(f"Target table: {config['target_path']}")
    print(f"Expected records: {config['expected_count']:,}")

total_expected = sum(
    config["expected_count"]
    for config in SOURCE_CONFIG.values()
)

print("\n" + "=" * 70)
print(f"Total source datasets: {len(SOURCE_CONFIG)}")
print(f"Total expected records: {total_expected:,}")
print("BRONZE INGESTION CONFIGURATION INITIALIZED")

COMMERCELAKE - BRONZE INGESTION CONFIGURATION
Catalog: commercelake
Target schema: bronze
Batch ID: batch_001

Entity: customers
Source format: csv
Source path: /Volumes/commercelake/landing/source_data/customers/batch_001/customers.csv
Target table: commercelake.bronze.raw_customers
Expected records: 1,000

Entity: products
Source format: csv
Source path: /Volumes/commercelake/landing/source_data/products/batch_001/products.csv
Target table: commercelake.bronze.raw_products
Expected records: 200

Entity: orders
Source format: json
Source path: /Volumes/commercelake/landing/source_data/orders/batch_001/orders.json
Target table: commercelake.bronze.raw_orders
Expected records: 5,000

Entity: order_items
Source format: parquet
Source path: /Volumes/commercelake/landing/source_data/order_items/batch_001/order_items.parquet
Target table: commercelake.bronze.raw_order_items
Expected records: 10,055

Entity: payments
Source format: json
Source path: /Volumes/commercelake/landing/source_data/

In [0]:
import uuid
from datetime import datetime, timezone

# Generate a unique pipeline execution ID
PIPELINE_RUN_ID = str(uuid.uuid4())

# Capture ingestion timestamp in UTC
INGESTION_TIMESTAMP = datetime.now(timezone.utc)

PIPELINE_NAME = "commercelake_bronze_initial_ingestion"

PIPELINE_STATUS = "RUNNING"

print("=" * 70)
print("COMMERCELAKE - PIPELINE RUN INITIALIZATION")
print("=" * 70)

print(f"Pipeline name: {PIPELINE_NAME}")
print(f"Pipeline run ID: {PIPELINE_RUN_ID}")
print(f"Batch ID: {BATCH_ID}")
print(f"Ingestion timestamp: {INGESTION_TIMESTAMP.isoformat()}")
print(f"Pipeline status: {PIPELINE_STATUS}")
print(f"Target layer: {BRONZE_SCHEMA}")
print(f"Total source datasets: {len(SOURCE_CONFIG)}")

print("\nPIPELINE RUN INITIALIZED SUCCESSFULLY")

COMMERCELAKE - PIPELINE RUN INITIALIZATION
Pipeline name: commercelake_bronze_initial_ingestion
Pipeline run ID: fa9ae030-86e6-41ab-870b-aaa16ae0a1e8
Batch ID: batch_001
Ingestion timestamp: 2026-11-26T06:27:16.044051+00:00
Pipeline status: RUNNING
Target layer: bronze
Total source datasets: 5

PIPELINE RUN INITIALIZED SUCCESSFULLY


In [0]:
from pyspark.sql import DataFrame

print("=" * 70)
print("COMMERCELAKE - BRONZE SOURCE DATA READING")
print("=" * 70)


def read_source_data(
    file_path: str,
    file_format: str
) -> DataFrame:

    if file_format == "csv":

        return (
            spark.read
            .option("header", "true")
            .option("inferSchema", "false")
            .option("mode", "FAILFAST")
            .csv(file_path)
        )

    elif file_format == "json":

        return (
            spark.read
            .option("multiLine", "false")
            .option("mode", "FAILFAST")
            .json(file_path)
        )

    elif file_format == "parquet":

        return spark.read.parquet(file_path)

    else:

        raise ValueError(
            f"Unsupported source format: {file_format}"
        )


source_dataframes = {}

for entity, config in SOURCE_CONFIG.items():

    print(f"\nReading entity: {entity}")

    df = read_source_data(
        file_path=config["source_path"],
        file_format=config["format"]
    )

    actual_count = df.count()
    expected_count = config["expected_count"]

    if actual_count != expected_count:

        raise ValueError(
            f"Record count mismatch for {entity}: "
            f"Expected {expected_count}, "
            f"Found {actual_count}"
        )

    source_dataframes[entity] = df

    print(f"Format: {config['format']}")
    print(f"Columns: {len(df.columns)}")
    print(f"Records: {actual_count:,}")
    print("Status: PASSED")


print("\n" + "=" * 70)

print(f"Total DataFrames initialized: {len(source_dataframes)}")

print("ALL SOURCE DATASETS READ SUCCESSFULLY")

COMMERCELAKE - BRONZE SOURCE DATA READING

Reading entity: customers
Format: csv
Columns: 13
Records: 1,000
Status: PASSED

Reading entity: products
Format: csv
Columns: 12
Records: 200
Status: PASSED

Reading entity: orders
Format: json
Columns: 11
Records: 5,000
Status: PASSED

Reading entity: order_items
Format: parquet
Columns: 15
Records: 10,055
Status: PASSED

Reading entity: payments
Format: json
Columns: 13
Records: 5,000
Status: PASSED

Total DataFrames initialized: 5
ALL SOURCE DATASETS READ SUCCESSFULLY


In [0]:
from pyspark.sql import functions as F

print("=" * 70)
print("COMMERCELAKE - BRONZE METADATA ENRICHMENT")
print("=" * 70)


def add_ingestion_metadata(
    df: DataFrame,
    source_path: str,
    batch_id: str,
    pipeline_run_id: str,
    ingestion_timestamp: datetime
) -> DataFrame:

    # Sort source columns for consistent record hashing
    source_columns = sorted(df.columns)

    # Generate SHA-256 hash using source columns only
    record_hash = F.sha2(
        F.to_json(
            F.struct(*[
                F.col(column)
                for column in source_columns
            ]),
            options={"ignoreNullFields": "false"}
        ),
        256
    )

    enriched_df = (
        df
        .withColumn(
            "_ingestion_timestamp",
            F.lit(ingestion_timestamp).cast("timestamp")
        )
        .withColumn(
            "_source_file",
            F.lit(source_path)
        )
        .withColumn(
            "_batch_id",
            F.lit(batch_id)
        )
        .withColumn(
            "_pipeline_run_id",
            F.lit(pipeline_run_id)
        )
        .withColumn(
            "_record_hash",
            record_hash
        )
    )

    return enriched_df


bronze_dataframes = {}

for entity, df in source_dataframes.items():

    enriched_df = add_ingestion_metadata(
        df=df,
        source_path=SOURCE_CONFIG[entity]["source_path"],
        batch_id=BATCH_ID,
        pipeline_run_id=PIPELINE_RUN_ID,
        ingestion_timestamp=INGESTION_TIMESTAMP
    )

    bronze_dataframes[entity] = enriched_df

    print(f"\nEntity: {entity}")
    print(f"Original columns: {len(df.columns)}")
    print(f"Enriched columns: {len(enriched_df.columns)}")
    print(
        f"Metadata columns added: "
        f"{len(enriched_df.columns) - len(df.columns)}"
    )
    print("Status: PASSED")


print("\n" + "=" * 70)
print(f"Total enriched DataFrames: {len(bronze_dataframes)}")
print("ALL BRONZE DATAFRAMES ENRICHED SUCCESSFULLY")

COMMERCELAKE - BRONZE METADATA ENRICHMENT

Entity: customers
Original columns: 13
Enriched columns: 18
Metadata columns added: 5
Status: PASSED

Entity: products
Original columns: 12
Enriched columns: 17
Metadata columns added: 5
Status: PASSED

Entity: orders
Original columns: 11
Enriched columns: 16
Metadata columns added: 5
Status: PASSED

Entity: order_items
Original columns: 15
Enriched columns: 20
Metadata columns added: 5
Status: PASSED

Entity: payments
Original columns: 13
Enriched columns: 18
Metadata columns added: 5
Status: PASSED

Total enriched DataFrames: 5
ALL BRONZE DATAFRAMES ENRICHED SUCCESSFULLY


In [0]:
print("=" * 70)
print("COMMERCELAKE - BRONZE DELTA TABLE INITIALIZATION")
print("=" * 70)

table_creation_results = []

for entity, df in bronze_dataframes.items():

    target_table = SOURCE_CONFIG[entity]["target_path"]

    table_exists = spark.catalog.tableExists(target_table)

    if not table_exists:

        (
            df.limit(0)
            .write
            .format("delta")
            .mode("ignore")
            .saveAsTable(target_table)
        )

        creation_status = "CREATED"

    else:

        creation_status = "ALREADY EXISTS"

    table_df = spark.table(target_table)

    actual_columns = len(table_df.columns)
    expected_columns = len(df.columns)

    # Compare column names and data types
    expected_schema = [
        (field.name, field.dataType.simpleString())
        for field in df.schema.fields
    ]

    actual_schema = [
        (field.name, field.dataType.simpleString())
        for field in table_df.schema.fields
    ]

    schema_matches = actual_schema == expected_schema

    if not schema_matches:

        raise RuntimeError(
            f"Schema mismatch for {target_table}"
        )

    table_creation_results.append({
        "entity": entity,
        "target_table": target_table,
        "creation_status": creation_status,
        "expected_columns": expected_columns,
        "actual_columns": actual_columns,
        "schema_matches": schema_matches
    })

    print(f"\nEntity: {entity}")
    print(f"Target table: {target_table}")
    print(f"Table status: {creation_status}")
    print(f"Expected columns: {expected_columns}")
    print(f"Actual columns: {actual_columns}")
    print(f"Schema matches: {schema_matches}")


print("\n" + "=" * 70)
print(f"Total Bronze tables initialized: {len(table_creation_results)}")
print("ALL BRONZE DELTA TABLES INITIALIZED SUCCESSFULLY")

COMMERCELAKE - BRONZE DELTA TABLE INITIALIZATION

Entity: customers
Target table: commercelake.bronze.raw_customers
Table status: CREATED
Expected columns: 18
Actual columns: 18
Schema matches: True

Entity: products
Target table: commercelake.bronze.raw_products
Table status: CREATED
Expected columns: 17
Actual columns: 17
Schema matches: True

Entity: orders
Target table: commercelake.bronze.raw_orders
Table status: CREATED
Expected columns: 16
Actual columns: 16
Schema matches: True

Entity: order_items
Target table: commercelake.bronze.raw_order_items
Table status: CREATED
Expected columns: 20
Actual columns: 20
Schema matches: True

Entity: payments
Target table: commercelake.bronze.raw_payments
Table status: CREATED
Expected columns: 18
Actual columns: 18
Schema matches: True

Total Bronze tables initialized: 5
ALL BRONZE DELTA TABLES INITIALIZED SUCCESSFULLY


In [0]:
print("=" * 70)
print("COMMERCELAKE - BRONZE INITIAL BATCH INGESTION")
print("=" * 70)

ingestion_results = []

for entity, df in bronze_dataframes.items():

    target_table = SOURCE_CONFIG[entity]["target_path"]
    expected_count = SOURCE_CONFIG[entity]["expected_count"]

    print(f"\nIngesting entity: {entity}")
    print(f"Target table: {target_table}")

    # Check whether the target table is empty
    before_count = spark.table(target_table).count()

    if before_count != 0:

        raise RuntimeError(
            f"Target table {target_table} is not empty. "
            "Initial ingestion stopped to prevent duplicates."
        )

    start_time = datetime.now(timezone.utc)

    # Append enriched source records into Delta table
    (
        df.write
        .format("delta")
        .mode("append")
        .saveAsTable(target_table)
    )

    end_time = datetime.now(timezone.utc)

    duration_seconds = (
        end_time - start_time
    ).total_seconds()

    # Verify records after ingestion
    after_count = spark.table(target_table).count()

    inserted_count = after_count - before_count

    status = (
        "PASSED"
        if inserted_count == expected_count
        else "FAILED"
    )

    ingestion_results.append({
        "entity": entity,
        "target_table": target_table,
        "batch_id": BATCH_ID,
        "pipeline_run_id": PIPELINE_RUN_ID,
        "expected_records": expected_count,
        "inserted_records": inserted_count,
        "duration_seconds": duration_seconds,
        "status": status
    })

    print(f"Records before ingestion: {before_count:,}")
    print(f"Expected records: {expected_count:,}")
    print(f"Inserted records: {inserted_count:,}")
    print(f"Records after ingestion: {after_count:,}")
    print(f"Duration: {duration_seconds:.2f} seconds")
    print(f"Status: {status}")

    if status == "FAILED":

        raise RuntimeError(
            f"Ingestion count mismatch for {entity}"
        )


total_inserted = sum(
    result["inserted_records"]
    for result in ingestion_results
)

total_duration = sum(
    result["duration_seconds"]
    for result in ingestion_results
)

print("\n" + "=" * 70)
print("BRONZE INGESTION SUMMARY")
print("=" * 70)

print(f"Batch ID: {BATCH_ID}")
print(f"Pipeline run ID: {PIPELINE_RUN_ID}")
print(f"Total tables processed: {len(ingestion_results)}")
print(f"Total records inserted: {total_inserted:,}")
print(f"Total write duration: {total_duration:.2f} seconds")

print("\nALL BRONZE DATASETS INGESTED SUCCESSFULLY")

COMMERCELAKE - BRONZE INITIAL BATCH INGESTION

Ingesting entity: customers
Target table: commercelake.bronze.raw_customers
Records before ingestion: 0
Expected records: 1,000
Inserted records: 1,000
Records after ingestion: 1,000
Duration: 2.31 seconds
Status: PASSED

Ingesting entity: products
Target table: commercelake.bronze.raw_products
Records before ingestion: 0
Expected records: 200
Inserted records: 200
Records after ingestion: 200
Duration: 1.66 seconds
Status: PASSED

Ingesting entity: orders
Target table: commercelake.bronze.raw_orders
Records before ingestion: 0
Expected records: 5,000
Inserted records: 5,000
Records after ingestion: 5,000
Duration: 1.55 seconds
Status: PASSED

Ingesting entity: order_items
Target table: commercelake.bronze.raw_order_items
Records before ingestion: 0
Expected records: 10,055
Inserted records: 10,055
Records after ingestion: 10,055
Duration: 1.68 seconds
Status: PASSED

Ingesting entity: payments
Target table: commercelake.bronze.raw_payment

In [0]:
print("=" * 70)
print("COMMERCELAKE - BRONZE RECORD COUNT VALIDATION")
print("=" * 70)

bronze_count_results = []

for entity, config in SOURCE_CONFIG.items():

    target_table = config["target_path"]
    expected_count = config["expected_count"]

    # Read persisted Delta table independently
    target_df = spark.table(target_table)

    # Count all records in the table
    total_count = target_df.count()

    # Count records belonging to the current batch
    batch_count = (
        target_df
        .filter(F.col("_batch_id") == BATCH_ID)
        .count()
    )

    difference = batch_count - expected_count

    status = (
        "PASSED"
        if batch_count == expected_count
        and total_count == expected_count
        else "FAILED"
    )

    bronze_count_results.append({
        "entity": entity,
        "target_table": target_table,
        "expected_count": expected_count,
        "total_count": total_count,
        "batch_count": batch_count,
        "difference": difference,
        "status": status
    })

    print(f"\nEntity: {entity}")
    print(f"Target table: {target_table}")
    print(f"Expected records: {expected_count:,}")
    print(f"Total table records: {total_count:,}")
    print(f"Current batch records: {batch_count:,}")
    print(f"Difference: {difference:,}")
    print(f"Status: {status}")


total_expected = sum(
    r["expected_count"]
    for r in bronze_count_results
)

total_actual = sum(
    r["total_count"]
    for r in bronze_count_results
)

total_batch_records = sum(
    r["batch_count"]
    for r in bronze_count_results
)

all_counts_valid = all(
    r["status"] == "PASSED"
    for r in bronze_count_results
)

print("\n" + "=" * 70)
print("BRONZE RECORD COUNT SUMMARY")
print("=" * 70)

print(f"Total expected records: {total_expected:,}")
print(f"Total Bronze records: {total_actual:,}")
print(f"Total current batch records: {total_batch_records:,}")
print(f"Total difference: {total_actual - total_expected:,}")

if all_counts_valid:

    print("\nALL BRONZE RECORD COUNT VALIDATIONS PASSED")

else:

    raise RuntimeError(
        "Bronze record count validation failed."
    )

COMMERCELAKE - BRONZE RECORD COUNT VALIDATION

Entity: customers
Target table: commercelake.bronze.raw_customers
Expected records: 1,000
Total table records: 1,000
Current batch records: 1,000
Difference: 0
Status: PASSED

Entity: products
Target table: commercelake.bronze.raw_products
Expected records: 200
Total table records: 200
Current batch records: 200
Difference: 0
Status: PASSED

Entity: orders
Target table: commercelake.bronze.raw_orders
Expected records: 5,000
Total table records: 5,000
Current batch records: 5,000
Difference: 0
Status: PASSED

Entity: order_items
Target table: commercelake.bronze.raw_order_items
Expected records: 10,055
Total table records: 10,055
Current batch records: 10,055
Difference: 0
Status: PASSED

Entity: payments
Target table: commercelake.bronze.raw_payments
Expected records: 5,000
Total table records: 5,000
Current batch records: 5,000
Difference: 0
Status: PASSED

BRONZE RECORD COUNT SUMMARY
Total expected records: 21,255
Total Bronze records: 2

In [0]:
print("=" * 70)
print("COMMERCELAKE - BRONZE METADATA AND SCHEMA VALIDATION")
print("=" * 70)

METADATA_COLUMNS = [
    "_ingestion_timestamp",
    "_source_file",
    "_batch_id",
    "_pipeline_run_id",
    "_record_hash"
]

metadata_validation_results = []

for entity, config in SOURCE_CONFIG.items():

    target_table = config["target_path"]

    df = spark.table(target_table)

    print(f"\nEntity: {entity}")
    print(f"Target table: {target_table}")

    # 1. Check required metadata columns
    missing_columns = [
        column for column in METADATA_COLUMNS
        if column not in df.columns
    ]

    if missing_columns:
        raise RuntimeError(
            f"Missing metadata columns: {missing_columns}"
        )

    # 2. Validate schema names and data types
    expected_schema = [
        (field.name, field.dataType.simpleString())
        for field in bronze_dataframes[entity].schema.fields
    ]

    actual_schema = [
        (field.name, field.dataType.simpleString())
        for field in df.schema.fields
    ]

    schema_matches = expected_schema == actual_schema

    # 3. Check null metadata values
    null_condition = F.lit(False)

    for column in METADATA_COLUMNS:
        null_condition = (
            null_condition | F.col(column).isNull()
        )

    null_metadata_count = df.filter(null_condition).count()

    # 4. Validate batch ID
    invalid_batch_count = df.filter(
        F.col("_batch_id") != BATCH_ID
    ).count()

    # 5. Validate pipeline run ID
    invalid_run_count = df.filter(
        F.col("_pipeline_run_id") != PIPELINE_RUN_ID
    ).count()

    # 6. Validate source file path
    invalid_source_count = df.filter(
        F.col("_source_file") != config["source_path"]
    ).count()

    # 7. Validate SHA-256 hash format
    invalid_hash_count = df.filter(
        ~F.col("_record_hash").rlike("^[a-f0-9]{64}$")
    ).count()

    # 8. Validate ingestion timestamp
    invalid_timestamp_count = df.filter(
        F.col("_ingestion_timestamp") !=
        F.lit(INGESTION_TIMESTAMP).cast("timestamp")
    ).count()

    # 9. Recompute record hash from source columns
    source_columns = sorted([
        column for column in df.columns
        if column not in METADATA_COLUMNS
    ])

    expected_hash = F.sha2(
        F.to_json(
            F.struct(*[
                F.col(column)
                for column in source_columns
            ]),
            options={"ignoreNullFields": "false"}
        ),
        256
    )

    hash_mismatch_count = df.filter(
        ~F.col("_record_hash").eqNullSafe(expected_hash)
    ).count()

    status = (
        "PASSED"
        if schema_matches and all([
            null_metadata_count == 0,
            invalid_batch_count == 0,
            invalid_run_count == 0,
            invalid_source_count == 0,
            invalid_hash_count == 0,
            invalid_timestamp_count == 0,
            hash_mismatch_count == 0
        ])
        else "FAILED"
    )

    metadata_validation_results.append({
        "entity": entity,
        "target_table": target_table,
        "schema_matches": schema_matches,
        "null_metadata_count": null_metadata_count,
        "invalid_batch_count": invalid_batch_count,
        "invalid_run_count": invalid_run_count,
        "invalid_source_count": invalid_source_count,
        "invalid_hash_count": invalid_hash_count,
        "invalid_timestamp_count": invalid_timestamp_count,
        "hash_mismatch_count": hash_mismatch_count,
        "status": status
    })

    print(f"Metadata columns present: {len(METADATA_COLUMNS)}")
    print(f"Schema matches: {schema_matches}")
    print(f"Records with null metadata: {null_metadata_count}")
    print(f"Invalid batch IDs: {invalid_batch_count}")
    print(f"Invalid pipeline run IDs: {invalid_run_count}")
    print(f"Invalid source paths: {invalid_source_count}")
    print(f"Invalid hash formats: {invalid_hash_count}")
    print(f"Invalid timestamps: {invalid_timestamp_count}")
    print(f"Record hash mismatches: {hash_mismatch_count}")
    print(f"Status: {status}")


all_metadata_valid = all(
    result["status"] == "PASSED"
    for result in metadata_validation_results
)

print("\n" + "=" * 70)

if all_metadata_valid:
    print("ALL BRONZE METADATA VALIDATIONS PASSED")
    print("ALL BRONZE SCHEMA VALIDATIONS PASSED")
else:
    raise RuntimeError(
        "Bronze metadata or schema validation failed."
    )

COMMERCELAKE - BRONZE METADATA AND SCHEMA VALIDATION

Entity: customers
Target table: commercelake.bronze.raw_customers
Metadata columns present: 5
Schema matches: True
Records with null metadata: 0
Invalid batch IDs: 0
Invalid pipeline run IDs: 0
Invalid source paths: 0
Invalid hash formats: 0
Invalid timestamps: 0
Record hash mismatches: 0
Status: PASSED

Entity: products
Target table: commercelake.bronze.raw_products
Metadata columns present: 5
Schema matches: True
Records with null metadata: 0
Invalid batch IDs: 0
Invalid pipeline run IDs: 0
Invalid source paths: 0
Invalid hash formats: 0
Invalid timestamps: 0
Record hash mismatches: 0
Status: PASSED

Entity: orders
Target table: commercelake.bronze.raw_orders
Metadata columns present: 5
Schema matches: True
Records with null metadata: 0
Invalid batch IDs: 0
Invalid pipeline run IDs: 0
Invalid source paths: 0
Invalid hash formats: 0
Invalid timestamps: 0
Record hash mismatches: 0
Status: PASSED

Entity: order_items
Target table: co

In [0]:
print("=" * 70)
print("COMMERCELAKE - DELTA TRANSACTION HISTORY")
print("=" * 70)

delta_history_results = []

for entity, config in SOURCE_CONFIG.items():

    target_table = config["target_path"]
    expected_count = config["expected_count"]

    print(f"\nEntity: {entity}")
    print(f"Target table: {target_table}")

    # Retrieve Delta transaction history
    history_df = spark.sql(
        f"DESCRIBE HISTORY {target_table}"
    )

    # Display relevant transaction information
    history_df.select(
        "version",
        "timestamp",
        "operation",
        "operationParameters",
        "operationMetrics"
    ).orderBy(
        F.col("version").asc()
    ).show(truncate=False)

    # Retrieve the latest committed transaction
    latest_transaction = (
        history_df
        .orderBy(F.col("version").desc())
        .first()
    )

    latest_version = latest_transaction["version"]
    latest_operation = latest_transaction["operation"]

    operation_metrics = (
        latest_transaction["operationMetrics"] or {}
    )

    written_records = int(
        operation_metrics.get("numOutputRows", -1)
    )

    status = (
        "PASSED"
        if latest_version >= 1
        and latest_operation == "WRITE"
        and written_records == expected_count
        else "FAILED"
    )

    delta_history_results.append({
        "entity": entity,
        "target_table": target_table,
        "latest_version": latest_version,
        "latest_operation": latest_operation,
        "written_records": written_records,
        "expected_records": expected_count,
        "status": status
    })

    print(f"Latest Delta version: {latest_version}")
    print(f"Latest operation: {latest_operation}")
    print(f"Expected records: {expected_count:,}")
    print(f"Transaction records written: {written_records:,}")
    print(f"Status: {status}")


all_history_valid = all(
    result["status"] == "PASSED"
    for result in delta_history_results
)

print("\n" + "=" * 70)
print("DELTA TRANSACTION HISTORY SUMMARY")
print("=" * 70)

print(f"Total tables inspected: {len(delta_history_results)}")

print(
    f"Total transaction records written: "
    f"{sum(r['written_records'] for r in delta_history_results):,}"
)

if all_history_valid:

    print("ALL DELTA TRANSACTION HISTORY VALIDATIONS PASSED")

else:

    raise RuntimeError(
        "Delta transaction history validation failed."
    )

COMMERCELAKE - DELTA TRANSACTION HISTORY

Entity: customers
Target table: commercelake.bronze.raw_customers
+-------+-----------------------+----------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [0]:
print("=" * 70)
print("COMMERCELAKE - FINAL BRONZE INGESTION REPORT")
print("=" * 70)

final_bronze_results = []

for entity, config in SOURCE_CONFIG.items():

    target_table = config["target_path"]

    ingestion_result = next(
        r for r in ingestion_results
        if r["entity"] == entity
    )

    count_result = next(
        r for r in bronze_count_results
        if r["entity"] == entity
    )

    metadata_result = next(
        r for r in metadata_validation_results
        if r["entity"] == entity
    )

    history_result = next(
        r for r in delta_history_results
        if r["entity"] == entity
    )

    # Retrieve physical Delta table information
    table_details = spark.sql(
        f"DESCRIBE DETAIL {target_table}"
    ).first()

    table_format = table_details["format"]
    number_of_files = table_details["numFiles"]
    size_in_bytes = table_details["sizeInBytes"]

    final_status = (
        "PASSED"
        if all([
            ingestion_result["status"] == "PASSED",
            count_result["status"] == "PASSED",
            metadata_result["status"] == "PASSED",
            history_result["status"] == "PASSED",
            table_format == "delta"
        ])
        else "FAILED"
    )

    final_bronze_results.append({
        "entity": entity,
        "target_table": target_table,
        "records": count_result["total_count"],
        "columns": len(spark.table(target_table).columns),
        "format": table_format,
        "delta_version": history_result["latest_version"],
        "number_of_files": number_of_files,
        "size_in_bytes": size_in_bytes,
        "write_duration_seconds": (
            ingestion_result["duration_seconds"]
        ),
        "status": final_status
    })


# Display consolidated verification report
final_report_df = spark.createDataFrame(
    final_bronze_results
)

display(
    final_report_df.select(
        "entity",
        "records",
        "columns",
        "format",
        "delta_version",
        "number_of_files",
        "size_in_bytes",
        "write_duration_seconds",
        "status"
    )
)


# Calculate final pipeline metrics
total_tables = len(final_bronze_results)

passed_tables = sum(
    r["status"] == "PASSED"
    for r in final_bronze_results
)

failed_tables = total_tables - passed_tables

total_records = sum(
    r["records"]
    for r in final_bronze_results
)

total_files = sum(
    r["number_of_files"]
    for r in final_bronze_results
)

total_storage_bytes = sum(
    r["size_in_bytes"]
    for r in final_bronze_results
)

total_write_duration = sum(
    r["write_duration_seconds"]
    for r in final_bronze_results
)

success_rate = (
    passed_tables / total_tables
) * 100

PIPELINE_STATUS = (
    "SUCCESS"
    if failed_tables == 0
    else "FAILED"
)

PIPELINE_END_TIMESTAMP = datetime.now(timezone.utc)

print("\n" + "=" * 70)
print("FINAL PIPELINE SUMMARY")
print("=" * 70)

print(f"Pipeline name: {PIPELINE_NAME}")
print(f"Pipeline run ID: {PIPELINE_RUN_ID}")
print(f"Batch ID: {BATCH_ID}")
print(f"Pipeline status: {PIPELINE_STATUS}")

print(f"\nTotal Bronze tables: {total_tables}")
print(f"Passed tables: {passed_tables}")
print(f"Failed tables: {failed_tables}")
print(f"Total Bronze records: {total_records:,}")
print(f"Total Delta data files: {total_files}")
print(f"Total storage bytes: {total_storage_bytes:,}")
print(f"Total storage MiB: {total_storage_bytes / 1024**2:.2f}")
print(f"Total write duration: {total_write_duration:.2f} seconds")
print(f"Success rate: {success_rate:.2f}%")

print(f"\nIngestion timestamp: {INGESTION_TIMESTAMP.isoformat()}")
print(f"Verification completed: {PIPELINE_END_TIMESTAMP.isoformat()}")

print("\n" + "=" * 70)

if PIPELINE_STATUS == "SUCCESS":

    print("BRONZE LAYER SUCCESSFULLY INITIALIZED")
    print("ALL INITIAL BATCH INGESTION VALIDATIONS PASSED")

else:

    raise RuntimeError(
        "Final Bronze ingestion verification failed."
    )

COMMERCELAKE - FINAL BRONZE INGESTION REPORT


entity,records,columns,format,delta_version,number_of_files,size_in_bytes,write_duration_seconds,status
customers,1000,18,delta,1,1,79062,2.306558,PASSED
products,200,17,delta,1,1,19025,1.656037,PASSED
orders,5000,16,delta,1,1,300953,1.552272,PASSED
order_items,10055,20,delta,1,1,571863,1.683482,PASSED
payments,5000,18,delta,1,1,345364,1.667156,PASSED



FINAL PIPELINE SUMMARY
Pipeline name: commercelake_bronze_initial_ingestion
Pipeline run ID: fa9ae030-86e6-41ab-870b-aaa16ae0a1e8
Batch ID: batch_001
Pipeline status: SUCCESS

Total Bronze tables: 5
Passed tables: 5
Failed tables: 0
Total Bronze records: 21,255
Total Delta data files: 5
Total storage bytes: 1,316,267
Total storage MiB: 1.26
Total write duration: 8.87 seconds
Success rate: 100.00%

Ingestion timestamp: 2026-11-26T06:27:16.044051+00:00
Verification completed: 2026-11-26T06:38:32.996666+00:00

BRONZE LAYER SUCCESSFULLY INITIALIZED
ALL INITIAL BATCH INGESTION VALIDATIONS PASSED
